# 26장 · 라우팅 · 분류 라벨을 검증하고 처리 함수 고르기

라우팅은 입력을 분류해 실행할 처리 함수를 고르는 기법입니다. 분류 결과의 허용값을 미리 정하면 프로그램에서 그 값으로 분기할 수 있습니다.

## 분류 라벨로 분기하기

라우팅은 입력을 분류한 뒤 그 결과에 맞는 처리 함수를 고르는 방법입니다. 짧은 문장을 분류해 `translate`가 돌아왔을 때만 번역을 요청합니다.

`ask(지시, 입력)`은 OpenAI SDK로 요청하고 답을 문자열로 돌려주는 수업용 함수입니다. 수업 커널에서 먼저 아래 셀로 함수를 정의하세요. 새 커널에서 openai를 찾지 못하면 `%pip install openai`를 실행하세요.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
text = "안녕을 영어로 번역해 줘."
label = ask("번역 요청은 translate, 그 밖에는 other 한 단어만 쓰세요.", text).strip()
print("분류 결과", repr(label))
if label == "translate":
    print(ask("번역한 단어만 쓰세요.", text))
elif label == "other":
    print("번역 요청이 아닙니다.")
else:
    print("분류 결과를 확인하세요.")

첫 줄에서 모델이 돌려준 라벨을 읽으세요. translate이면 번역을 한 번 더 요청합니다. 입력을 “반가워!”로 바꿔 other가 나오면 안내 문구만 출력합니다. 두 라벨에 속하지 않는 응답은 마지막 else에서 확인할 수 있습니다.

## 응용 실습 · 문의 분류와 처리

### 앞 장에서 쓴 답을 검토 요청으로 바꾸기

25장에서 저장한 초안을 읽어 “이 안내를 검토해 주세요”라는 요청을 만듭니다. 아래 세 갈래 분기에서 이 요청이 어디로 가는지 예측해 보세요. 앞 장을 건너뛰면 수업 묶음에 들어 있는 복귀 예시를 읽습니다. 첫 출력에 자료의 출처가 표시됩니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

previous = read_previous(root, 25)
review_message = "이 초안이 규정에 맞는지 검토해 주세요.\n" + previous["draft"]
print(review_message)

교재 복귀 예시 · chapter25.json
이 초안이 규정에 맞는지 검토해 주세요.
6월 11일 정오까지 원래 신청 게시글에 연장 요청 댓글을 남기세요. 담당자가 승인 댓글을 달아야 반납일이 바뀝니다.


## 처리 함수가 받을 세 라벨 정하기

`select_route`는 문의 하나를 읽고 `answer`, `review`, `clarify` 중 하나를 반환합니다. 규정으로 바로 답할 질문, 이미 쓴 답의 검토 요청, 신청 정보가 빠진 요청을 이 순서로 구분합니다. 예를 들어 “제가 쓴 수령 안내가 맞나요?”의 기대 라벨은 `review`입니다.

## 응용에 사용할 자료 열기

실습 묶음의 압축을 풀고 노트북과 `pre-mid` 폴더를 나란히 두세요. 다음 셀은 대여 규정 파일을 읽어 `rules`에 저장합니다. 응용에서도 앞에서 정의한 `ask`를 사용합니다.

In [ ]:
from pathlib import Path

rules = (next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pre-mid/documents/sources/loan-rules.md").is_file()) / "pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")

In [ ]:
incomplete = '대여 신청 좀 부탁드려요. 이번 달에 쓸 예정이에요.'
review_request = '신입 회원에게 올릴 이 안내가 규정에 맞는지 봐 주세요. ‘학생증을 친구에게 맡기면 대신 받을 수 있고, 기기는 이틀 뒤까지 가져오면 됩니다.’'
ordinary = "승인받은 회원은 어디에서 녹음기를 받나요?"

## select_route에서 허용한 라벨만 반환하기

모델이 돌려준 글을 `strip().lower()`로 정리한 뒤 허용한 세 문자열에 포함되는지 확인합니다. 예를 들어 `" Review "`는 `"review"`가 됩니다. `"review please"`는 허용한 값과 다르므로 `raise ValueError(...)`에서 멈추고 처리 함수를 호출하지 않습니다.

라우팅은 입력을 어느 처리 단계로 보낼지 정하는 기법입니다. `select_route(message)`는 모델이 쓴 라벨 한 개를 받아 `strip().lower()`로 양끝 공백과 대소문자를 정리합니다. 세 허용값에 있는지 검사한 뒤 문자열을 돌려줍니다. 가령 응답이 “분류 결과는 review입니다”라면 정리해도 허용값과 다릅니다. 그때는 처리 함수를 부르기 전에 예외가 납니다.

In [ ]:
route_instruction = """작업 이름 하나만 쓰세요.
answer: 규정을 읽어 답할 수 있는 질문.
review: 사용자가 작성한 안내문을 규정과 대조해 달라는 요청.
clarify: 신청에 필요한 기종 또는 정확한 날짜를 사용자가 주지 않음.
예약을 대신 처리하지 마세요."""
def select_route(message):
    chosen = ask(route_instruction, message).strip().lower()
    # 예기치 않은 이름은 기록하고 작성 단계로 넘기지 않습니다.
    if chosen not in {"answer", "review", "clarify"}:
        raise ValueError("작업 이름 확인 필요: " + chosen)
    return chosen

### 라벨을 정리하는 두 메서드

In [ ]:
raw_label = " Review\n"
clean_label = raw_label.strip().lower()
print(repr(raw_label))
print(clean_label in {"answer", "review", "clarify"})

' Review\n'
True


첫 줄의 공백과 줄바꿈은 `repr`로 보면 드러납니다. 둘째 줄이 True인 이유는 정리한 값이 “review”이기 때문입니다. “review.”로 바꾸면 마침표가 남아 같은 검사에서 False가 됩니다.

In [ ]:
route = select_route(incomplete)
print(route)

Q3에서 clarify가 나왔다면 실제 요청을 살펴보세요. “이번 달”은 기종도 특정 대여일도 알려 주지 않습니다. 반대로 “대여일은 언제까지 적나요?”는 날짜를 알려 달라는 신청이 아니라 규정 질문입니다. 이 차이를 분류 지시에 넣을지 새 입력으로 판단하세요.

> **R1**　동아리 회원은 대여일 전날 오후 5시까지 게시판에 기종(A형 또는 B형)과 대여일을 적어 신청합니다. 대여 담당자가 게시글에 승인 댓글을 달면 예약이 끝납니다.

### 실습 · 낯선 라벨을 처리 함수 앞에서 걸러내기

모델 응답을 흉내 낸 문자열 두 개가 있습니다. 공백과 대소문자를 정리하고, 허용된 작업 이름인지 출력하세요. “Review please”를 잘못 review 경로로 보내지 않아야 합니다.

In [ ]:
labels = [" Review ", "Review please"]
# 각 응답을 정리하고 answer, review, clarify 중 하나인지 검사하세요.

<details><summary>정답 코드와 실행 결과</summary><pre><code>labels = [&quot; Review &quot;, &quot;Review please&quot;]
for raw in labels:
    label = raw.strip().lower()
    print(label, label in {&quot;answer&quot;, &quot;review&quot;, &quot;clarify&quot;})</code></pre><pre>review True
review please False</pre><p>strip과 lower를 적용해도 두 번째 값에는 please가 남습니다. 실제 select_route는 허용 목록 밖의 값을 받으면 오류를 내고 멈춥니다.</p></details>

## handle로 선택된 분기만 실행하기

각 분기의 끝에서 바로 결과를 돌려줍니다. 추가 질문으로 끝난 뒤 공통 작성 코드를 또 실행하면 정보가 없는데도 답안이 하나 더 생깁니다. 아래 검토 경로는 잘못된 인용을 찾아 수정안을 쓰고, 일반 경로는 규정에 따라 답합니다.

`handle(route, message)`는 두 값을 받습니다. `route`로 실행할 분기를 고르고 `message`로 그 분기에 넘길 원문을 보존합니다. 예를 들어 `clarify`가 들어오면 빠진 정보를 묻는 요청 한 번을 보낸 뒤 `return`으로 끝납니다. 다른 두 분기는 이번 호출에서 실행되지 않습니다.

In [ ]:
def handle(route, message):
    if route == "clarify":
        return ask("빠진 신청 정보만 짧게 물으세요. 신청이나 예약을 확정하지 마세요.", rules + "\n" + message)
    if route == "review":
        return ask("초안의 잘못된 문구를 인용하고 조항과 수정안을 보여 주세요.", rules + "\n" + message)
    if route == "answer":
        return ask("대여 규정에 따라 질문에 직접 답하고 조항을 적으세요.", rules + "\n" + message)
    raise ValueError("허용되지 않은 작업 이름")

In [ ]:
print(handle(route, incomplete))

올바른 추가 질문은 기종과 정확한 날짜를 묻습니다. “신청이 완료되었습니다”라는 말은 프로그램이 하지 않은 일을 약속하므로 문제가 됩니다. 판단과 실제로 수행한 행동을 나란히 보세요.

In [ ]:
chosen = select_route(review_request)
print(chosen)
print(handle(chosen, review_request))

Q2는 규정을 묻는 듯 보이지만 “이 안내가 맞는지” 검토해 달라고 했습니다. 결과가 review여도 수정할 두 구절이 모두 나왔는지 별도로 확인하세요. 작업 선택과 작업 결과의 정확성은 서로 다른 질문입니다.

> **Q2**　“신입 회원에게 올릴 이 안내가 규정에 맞는지 봐 주세요. ‘학생증을 친구에게 맡기면 대신 받을 수 있고, 기기는 이틀 뒤까지 가져오면 됩니다.’”

> **R2**　승인받은 회원은 대여일 오후 3시부터 5시까지 가온관 1층 안내 데스크에서 기기를 받습니다. 학생증을 보여 주세요. 친구가 대신 받을 수 없습니다.

> **R3**　수령 다음 날 오후 5시까지 가온관 1층 안내 데스크로 반납해 주세요. 기기와 함께 빌린 케이블도 가져오세요.

## 작업 이름을 읽으면 어디로 갈까

```mermaid
flowchart TD
    A["회원 문의"] --> B["작업 이름 제안"]
    B --> C{"허용된 이름인가?"}
    C -->|아니요| D["오류 확인"]
    C -->|answer| E["규정 답변"]
    C -->|review| F["문구 검토"]
    C -->|clarify| G["빠진 정보 질문"]
```

“이번 달에 B형 신청해 주세요”를 받았을 때 B형은 이미 압니다. clarify가 오면 날짜를 묻고 끝납니다. 답변·검토 함수에 같은 입력을 차례로 넣지 않습니다. 이름이 셋 중 하나가 아니면 select_route에서 멈춥니다.

## 분류와 회신을 따로 대조하기

**집필자가 기대한 예시**　Q3의 경로는 clarify이고, 회신은 “필요한 기종이 A형인가요, B형인가요? 정확한 대여일을 알려 주세요.”입니다. “이번 달”을 구체적인 날짜로 바꾸어 말하면 신청자 확인 없이 일정이 정해집니다. Q2의 경로는 review이고, 회신에는 “친구에게 맡기면”과 “이틀 뒤” 두 원문 구절이 있어야 합니다. 이 두 구절 중 하나만 나온 경우 올바른 경로에서도 검토 품질이 부족할 수 있습니다.

## 자료로 해결할 빈칸과 사람에게 물을 빈칸

회원이 “오후 몇 시에 찾아가야 해요?”라고 물으면 규정에서 시간을 찾을 수 있습니다. 반면 “다음 주쯤 쓰고 싶은데 신청해 주세요”에서는 정확한 날이 사용자의 선택입니다. 화면에 “무엇을 더 알려 주세요?”를 매번 띄우면 회원은 규정에서 답할 수 있는 일까지 떠맡게 됩니다. 둘이 짝을 지어 문의에 밑줄을 긋고 원문 조항, 신청자, 안내문 작성자 중 누가 빈칸을 채울 수 있는지 먼저 적으세요. 분류 결과가 달라질 때 근거가 됩니다.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 실습 · 비슷하게 들리는 세 문장 분류하기

개인 실습 90분. (가) “연장 신청에 무엇이 필요하죠?” (나) “이번 달에 B형 신청해 주세요.” (다) “안내문에 ‘B형은 친구가 수령 가능’이라고 썼는데 그대로 올려도 돼요?”에 예상 경로를 먼저 붙이세요. 같은 뜻을 구어체로 바꿔 한 번 더 보냅니다. 다르게 분류된 한 쌍 또는 가장 헷갈린 두 문의의 차이를 원문에서 찾아 지시를 다듬고 재실행하세요.

In [ ]:
message = "이번 달에 B형 신청해 주세요."
# 기종은 알려 줬지만 날짜는 특정되지 않았습니다.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>for message in [&quot;연장 신청에 무엇이 필요하죠?&quot;, &quot;늘리려면 뭘 해야 하죠?&quot;,
                &quot;이번 달에 B형 신청해 주세요.&quot;, &quot;B형 좀 빌려줘요. 이달 중에요.&quot;,
                &quot;‘B형은 친구가 수령 가능’이라고 썼는데 그대로 올려도 돼요?&quot;,
                &quot;친구가 가져가도 된다고 공지하려는데 맞을까요?&quot;]:
    selected = select_route(message)
    print(message, selected)
    print(handle(selected, message))</code></pre>
<p>(가)는 answer, (나)는 clarify, (다)는 review가 적합합니다. (나)의 추가 질문은 날짜 하나면 됩니다. 기종까지 다시 물었다면 경로는 옳고 질문의 범위가 넓습니다. 경로명만 채점하면 이 문제를 놓칩니다.</p>
</details>

## 검토 담당자에게 초안 넘기기

앞에서 읽은 review_message를 분류하세요. review가 나왔을 때만 초안을 검토 담당자에게 넘깁니다. 다른 라벨이면 분류 지시를 고치고 이 셀을 다시 실행하세요.

In [ ]:
route_for_draft = select_route(review_message)
if route_for_draft != "review":
    raise ValueError(f"예상 review, 실제 {route_for_draft}. 검토 요청과 분류 지시를 확인하세요.")
record = {"question": previous["question"], "draft": previous["draft"], "route": route_for_draft}
save_record(root, 26, record)

선택된 경로를 읽으세요. 다음 검토자는 route와 draft를 함께 받아야 검토 대상을 알 수 있습니다. 실시간 모델을 실행하지 않았으면 아래 셀만 골라 실행해 교재 예시의 자료 구조를 읽을 수 있습니다. 첫 줄이 파일의 출처입니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

saved = read_previous(root, 26)
print(saved["route"], saved["draft"])

교재 복귀 예시 · chapter26.json
review 6월 11일 정오까지 원래 신청 게시글에 연장 요청 댓글을 남기세요. 담당자가 승인 댓글을 달아야 반납일이 바뀝니다.


## 다음 작업에 가져갈 기록

예상 경로와 실제 경로가 달랐던 문장, 그 이유를 설명하는 표현을 보관하세요. 다음 장에서는 review 경로를 사실 검토와 표현 검토로 나눕니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)